In [1]:
import os
import pickle

### Settings

In [2]:
DIR_DATA = "data"
DIR_TEXTS = os.path.join(DIR_DATA, "texts")
FILE_METADATA = os.path.join(DIR_DATA, "metadata.csv")

In [3]:
# Load the cleaned corpus
with open(os.path.join(DIR_DATA, "cleaned_corpus.txt"), "r", encoding="utf-8") as f:
    corpus = f.read()
    
tokens = corpus.split()
print(f"Number of tokens: {len(tokens)}")

Number of tokens: 12207411


In [4]:
print(f"Number of unique tokens: {len(set(tokens))}")

Number of unique tokens: 25280


### Bi-gram

In [5]:
import gzip

PROGRESS_RESOLUTION = 50000
PROGRESS_BAR_WIDTH = 50
NS = [2, 3, 4, 5]

# Converting tokens to integers drastically reduces memory and file size
print("Building vocabulary...")
vocab = sorted(list(set(tokens)))
token_to_idx = {token: i for i, token in enumerate(vocab)}
idx_to_token = {i: token for i, token in enumerate(vocab)}

print("Encoding tokens...")
token_ids = [token_to_idx[t] for t in tokens]

def get_ngram_counts(input_tokens: list, ns: list[int]):

    # Allows us to safely start the loop at 1
    assert all(n >= 2 for n in ns), "n-grams only make sense for n >= 2"
    
    # Big dictionary of dictionaries
    ngram_counts = {
        n: {} for n in ns
    }
    
    # Iterate through each token in the corpus (skip first n-1 tokens)
    for token_i in range(1, len(input_tokens)):
        
        # Print progress every PROGRESS_RESOLUTION tokens
        if token_i % PROGRESS_RESOLUTION == 0:
            progress = token_i / len(input_tokens)
            bar = "#" * int(progress * PROGRESS_BAR_WIDTH)
            print(f"[{bar:<{PROGRESS_BAR_WIDTH}}] {progress:.2%} ({token_i}/{len(input_tokens)})", end="\r")
            
        for n in ns:
            
            # Skip if not enough tokens past
            if token_i < n - 1:
                continue
            
            # Extract the n-gram
            n_gram = tuple(input_tokens[token_i - n + 1: token_i + 1])
            ngram_counts[n][n_gram] = ngram_counts[n].get(n_gram, 0) + 1

    print(f"[{'#' * PROGRESS_BAR_WIDTH}] 100.00% ({len(input_tokens)}/{len(input_tokens)})")
    return ngram_counts

# Use encoded tokens
ngrams = get_ngram_counts(token_ids, NS)

# Pickle the n-grams
# We store the mapping so we can reconstruct the strings later
file_name = f"ngrams{'-'.join(map(str, NS))}.pkl"
print("Saving pickle...")
with open(os.path.join(DIR_DATA, file_name), "wb") as f:
    pickle.dump({
        "ngrams": ngrams,
        "idx_to_token": idx_to_token
    }, f)
print(f"Saved to {os.path.join(DIR_DATA, file_name)}")

Building vocabulary...
Encoding tokens...
[##################################################] 100.00% (12207411/12207411)
Saving compressed pickle...
Saved to data\ngrams2-3-4-5.pkl
